# Fencing

In [19]:
import numpy as np
import matplotlib.pylab as plt

## Preprocessing

In [94]:
with open("inputs/12", "r") as fp:
    data = fp.read()
    
data = data.split()

In [36]:
data = """RRRRIICCFF
RRRRIICCCF
VVRRRCCFFF
VVRCCCJFFF
VVVVCJJCFE
VVIVCCJJEE
VVIIICJJEE
MIIIIIJJEE
MIIISIJEEE
MMMISSJEEE""".split()

In [95]:
l0, l1 = len(data), len(data[0])

M = np.zeros((l0+2, l1+2)).astype(int).astype(str)

for idx, line in enumerate(data):
    for jdx, c in enumerate(line):
        M[idx+1, jdx+1] = c

# Part 1

1. Isolate groups
2. Compute area
3. Compute perimeter

##### Visit matrix

In [96]:
V = np.zeros(M.shape) 
V[1:-1, 1:-1] = 1

def visit_map(a0, b0, V):
    """
    :param a0, b0: reference coordinate
    :param V: visit matrix
    """
    group = []
    lst = [(a0, b0)]
    c = M[a0, b0]
    V[a0, b0] = 0
    while lst != []:
        a, b = lst.pop()
        group.append((a, b))
        
        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            a1, b1 = a + dx, b + dy
            if (M[a1, b1] == c) & (V[a1, b1] == 1):
                lst.append((a1, b1))
                V[a1, b1] = 0
    
    return group

### Isolate each group

In [97]:
lst_groups = []
while V.sum() > 0:
    # Select one point that has not been visited
    a, b = np.array(np.where(V == 1)).T[0]
    gp = visit_map(a, b, V)
    lst_groups.append(gp)


### Compute perimeter

For each element of the group, check if up/left/right/down belong to the group or not.
If not, add 1 to the perimeter

In [40]:
def get_perimeter(group):
    a0, b0 = group[0]
    c = M[a0, b0]
    cnt = 0
    for a, b in group:
        for dx, dy in [(0, 1), (0, -1), (1, 0), (-1, 0)]:
            a1, b1 = a + dx, b + dy
            if M[a1, b1] != c:
                cnt += 1
    return cnt

### Get full score

In [41]:
S = 0
for gp in lst_groups:
    area = len(gp)
    perim = get_perimeter(gp)
    S += area * perim

In [35]:
print(S)

1437300


# Part 2

Instead of perimeter, compute side.

How to get side?

1. Select an element.
2. It should not be surrounded by only similar item, ie, at least one dissimilar item.
3. Given dissimilar items, iterate in the neighborhood to mark it as visited

In [102]:
def get_block_size(gp):
    X = np.array(gp)
    m0, M0 = X[:, 0].min(), X[:, 0].max()

    # Identify blocks at m=m0
    lst0 = []

    cnt = 0
    for m in range(m0, M0+2):
        lst1 = X[X[:, 0] == m, 1].tolist()
        c  = count_new_edges(lst0, lst1)
        cnt += c
        lst0 = lst1
        
    return cnt

In [103]:
def count_new_edges(lst0, lst1):
    cnt = 0

    # Check new added element 
    # Array represent new block
    M0 = max(max(lst1 + [0]), max(lst0 + [0]))
    
    arr = np.zeros((2, M0+2))
    arr[0, lst0] = 1
    arr[1, lst1] = 1

    # Check vertical borders
    for i in range(M0+1):
        if arr[1, i] == arr[1, i+1]:
            # No border on second level
            continue
        elif (arr[1, i] == arr[0, i]) & (arr[1, i+1] == arr[0, i+1]):
            # Border, but same as above:
            continue
        else:
            # True vertical border !
            cnt += 1
            if arr[1, i] == 1:
                cnt += 1
                
    for i in range(M0+1):
        if arr[0, i] == arr[0, i+1]:
            # No border on firt level
            continue
        elif (arr[1, i] == arr[0, i]) & (arr[1, i+1] == arr[0, i+1]):
            # Border, but same as before:
            continue
        else:
            # True vertical border !
            if arr[0, i] == 0:
                cnt += 1
        
    
    return cnt

In [ ]:
S = 0
for gp in lst_groups:
    area = len(gp)
    perim = get_block_size(gp)
    S += area * perim

print(S)